# Customer Intelligence System using Classification, Ensemble & Clustering

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Clustering
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.metrics import silhouette_score

# Classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

import warnings
warnings.filterwarnings('ignore')


## 2. Load the Data & First Look

In [ ]:
df = pd.read_csv("Country-data.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df['country'].value_counts().sum()

In [ ]:
df.info()

In [ ]:
df.describe()

### 2.1 Check for Missing Values & Duplicates

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

**Observation:** The dataset has **167 rows and 10 columns** (`country` + 9 numeric socio-economic/health indicators), with **no missing values** and **no duplicate rows**, so it is clean and ready for analysis.

## 3. Exploratory Data Analysis (EDA)

### 3.1 Univariate Analysis — Distribution of Features

In [ ]:
num_cols = df.columns[1:]

plt.figure(figsize=(16, 12))
for i, col in enumerate(num_cols, 1):
    plt.subplot(3, 3, i)
    sns.histplot(df[col], kde=True, color='teal')
    plt.title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

**Observation:** Most features (`child_mort`, `exports`, `income`, `inflation`, `gdpp`) are **right-skewed** — a small number of countries have very high values (e.g., very high income/GDP per capita) while most countries cluster at lower values. `life_expec` is left-skewed, while `health` and `total_fer` are closer to a normal distribution.

### 3.2 Correlation Analysis

In [ ]:
plt.figure(figsize=(10, 8))
corr = df.iloc[:, 1:].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap of Socio-Economic Features')
plt.show()

**Observation:**
- `income` and `gdpp` are **strongly positively correlated** (~0.9), both reflecting a country's wealth.
- `child_mort` and `total_fer` are **strongly positively correlated** (~0.85) — countries with higher fertility rates tend to have higher child mortality.
- `child_mort` is **strongly negatively correlated** with `life_expec` (~-0.89) — higher child mortality is associated with lower life expectancy.
- `exports` and `imports` are also positively correlated (~0.74), as expected for trade-dependent economies.

These strongly correlated groups will likely drive the cluster separations we see later.

### 3.3 Outlier Detection

In [ ]:
plt.figure(figsize=(16, 10))
for i, col in enumerate(num_cols, 1):
    plt.subplot(3, 3, i)
    sns.boxplot(x=df[col], color='salmon')
    plt.title(col)
plt.tight_layout()
plt.show()

**Observation:** `income`, `gdpp`, and `child_mort` show several high-value outliers — these correspond to very wealthy nations (e.g., Qatar, Luxembourg) and very low-income/high-mortality nations (e.g., several African countries). These are **genuine, meaningful extreme values** (not data errors), so they are retained — they are exactly the kind of extreme "customers" a segmentation/intelligence system needs to identify.

## 4. Feature Scaling

Clustering algorithms (K-Means, DBSCAN, Hierarchical) and distance-based classifiers are sensitive to feature scale. Since features like `income`/`gdpp` (in thousands) and `inflation`/`total_fer` (single digits) are on very different scales, we standardize all numeric features using `StandardScaler` (zero mean, unit variance) before modelling.

In [ ]:
X = df.drop(columns='country')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

scaled_df = pd.DataFrame(X_scaled, columns=X.columns)
scaled_df.head()

In [ ]:
scaled_df.shape

## 5. Clustering — Unsupervised Customer Segmentation

We now segment the 167 "customers" (countries) into meaningful groups based on their socio-economic profile, using three clustering algorithms: **K-Means**, **Hierarchical Clustering**, and **DBSCAN**.

### 5.1 K-Means Clustering

#### 5.1.1 Elbow Method — choosing the number of clusters

In [ ]:
wcss = []

for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), wcss, marker='o')
plt.xlabel("Number of Clusters (K)")
plt.ylabel("WCSS (Inertia)")
plt.title("Elbow Method for Optimal K")
plt.show()

#### 5.1.2 Silhouette Score — confirming the optimal K

In [ ]:
sil_scores = []

for i in range(2, 11):
    kmeans = KMeans(n_clusters=i, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, labels))

plt.figure(figsize=(8, 5))
plt.plot(range(2, 11), sil_scores, marker='o', color='green')
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score vs Number of Clusters")
plt.show()

**Observation:** The elbow in the WCSS curve appears around **K = 3**, and the silhouette score is also highest/relatively high at **K = 3**. We therefore proceed with **K = 3 clusters**, consistent with grouping countries into broad development tiers (e.g., *developed*, *developing*, and *under-developed/needs-aid*).

#### 5.1.3 Final K-Means Model (K = 3)

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['KMeans_Cluster'] = kmeans.fit_predict(X_scaled)

df['KMeans_Cluster'].value_counts()

#### 5.1.4 Visualizing the Clusters (PCA Projection)

In [ ]:
pca = PCA(n_components=2, random_state=42)
pca_components = pca.fit_transform(X_scaled)

df['PCA1'] = pca_components[:, 0]
df['PCA2'] = pca_components[:, 1]

plt.figure(figsize=(8, 6))
sns.scatterplot(x='PCA1', y='PCA2', hue='KMeans_Cluster', data=df,
                palette='Set2', s=70, edgecolor='k')
plt.title('K-Means Clusters (PCA Projection)')
plt.show()

print(f"Variance explained by PCA1 & PCA2: {pca.explained_variance_ratio_.sum():.2%}")

#### 5.1.5 Cluster Profiling

In [ ]:
cluster_profile = df.groupby('KMeans_Cluster')[num_cols].mean().round(2)
cluster_profile

In [ ]:
for c in sorted(df['KMeans_Cluster'].unique()):
    print(f"Cluster {c} sample countries:", df[df['KMeans_Cluster'] == c]['country'].sample(5, random_state=1).tolist())

In [ ]:
segment_map = {0: 'Developed / High-Value', 1: 'Under-developed / Needs Support', 2: 'Developing / Mid-Value'}
df['Segment'] = df['KMeans_Cluster'].map(segment_map)
df[['country', 'Segment']].sample(10, random_state=1)

### 5.2 Hierarchical Clustering

We cross-check the K-Means segmentation with **Agglomerative (Hierarchical) Clustering** using Ward's linkage.

In [ ]:
plt.figure(figsize=(14, 6))
linked = linkage(X_scaled, method='ward')
dendrogram(linked, truncate_mode='lastp', p=20, leaf_rotation=90)
plt.title('Hierarchical Clustering Dendrogram (last 20 merges)')
plt.xlabel('Cluster size / Sample index')
plt.ylabel('Ward Distance')
plt.show()

In [ ]:
hc = AgglomerativeClustering(n_clusters=3, linkage='ward')
df['Hierarchical_Cluster'] = hc.fit_predict(X_scaled)

df['Hierarchical_Cluster'].value_counts()

In [ ]:
print("Silhouette Score (K-Means):     ", round(silhouette_score(X_scaled, df['KMeans_Cluster']), 3))
print("Silhouette Score (Hierarchical):", round(silhouette_score(X_scaled, df['Hierarchical_Cluster']), 3))

**Observation:** The dendrogram also shows **3 well-separated high-level groups**, supporting the choice of K = 3. K-Means achieves a slightly higher silhouette score, so it remains our primary segmentation, while hierarchical clustering visually confirms the same broad grouping structure.

### 5.3 DBSCAN (Density-Based Clustering)

DBSCAN groups points that are densely packed together and marks isolated points as **noise/outliers** — useful for flagging unusual "customers" that don't fit any mainstream segment.

#### 5.3.1 Choosing `eps` via the K-Distance Graph

In [ ]:
neighbors = NearestNeighbors(n_neighbors=5)
neighbors_fit = neighbors.fit(X_scaled)
distances, _ = neighbors_fit.kneighbors(X_scaled)

k_distances = np.sort(distances[:, 4])

plt.figure(figsize=(8, 5))
plt.plot(k_distances)
plt.title('K-Distance Graph (5th Nearest Neighbour)')
plt.xlabel('Points sorted by distance')
plt.ylabel('5th Nearest Neighbour Distance')
plt.axhline(y=1.2, color='red', linestyle='--', label='chosen eps = 1.2')
plt.legend()
plt.show()

#### 5.3.2 Fitting DBSCAN

In [ ]:
dbscan = DBSCAN(eps=1.2, min_samples=5)
df['DBSCAN_Cluster'] = dbscan.fit_predict(X_scaled)

df['DBSCAN_Cluster'].value_counts()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x='PCA1', y='PCA2', hue='DBSCAN_Cluster', data=df,
                palette='Set1', s=70, edgecolor='k')
plt.title('DBSCAN Clusters (PCA Projection) — Cluster -1 = Noise/Outliers')
plt.show()

In [ ]:
df.groupby('DBSCAN_Cluster')[num_cols].mean().round(2)

**Observation:** DBSCAN finds **3 dense clusters** (broadly matching the "Developed", "Developing" and "Under-developed" groups from K-Means) plus **53 "noise" countries (-1)** with mean indicators that fall *between* the dense groups — i.e., transitional/borderline economies that don't fit neatly into one segment. This is a useful additional signal: these borderline countries may need individual review rather than a blanket segment-level strategy.


We proceed with the **K-Means segments (`KMeans_Cluster` / `Segment`)** as the ground-truth label for the classification stage, since they give a clean, balanced 3-class target that aligns with a real-world "customer tiering" use case.

## 6. Classification — Predicting the Customer Segment

Re-running a full clustering pipeline every time a *new* customer (country) arrives is expensive. Instead, we train a **classifier** on the segments discovered above, so that **new records can be instantly assigned to the correct segment** from their raw features — this is the predictive core of the "Customer Intelligence System".

### 6.1 Train-Test Split

In [ ]:
X_clf = scaled_df.copy()
y_clf = df['KMeans_Cluster']

X_train, X_test, y_train, y_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)
print("\nClass distribution in train:\n", y_train.value_counts(normalize=True).round(2))

### 6.2 Helper Function for Training & Evaluation

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    print(f"--- {name} ---")
    print(classification_report(y_test, y_pred, digits=3))

    return {'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1}, model

### 6.3 Baseline Classification Models

Before moving to ensembles, we establish baselines using **Logistic Regression** and a single **Decision Tree** .

In [ ]:
results_list = []

res, _ = evaluate_model(LogisticRegression(max_iter=1000, random_state=42),
                         X_train, X_test, y_train, y_test, 'Logistic Regression')
results_list.append(res)

res, _ = evaluate_model(DecisionTreeClassifier(random_state=42),
                         X_train, X_test, y_train, y_test, 'Decision Tree')
results_list.append(res)

### 6.4 Ensemble Classification Models


In [ ]:
res, rf_model = evaluate_model(RandomForestClassifier(n_estimators=100, random_state=42),
                                X_train, X_test, y_train, y_test, 'Random Forest')
results_list.append(res)

res, ada_model = evaluate_model(AdaBoostClassifier(random_state=42),
                                 X_train, X_test, y_train, y_test, 'AdaBoost')
results_list.append(res)

res, gb_model = evaluate_model(GradientBoostingClassifier(random_state=42),
                                X_train, X_test, y_train, y_test, 'Gradient Boosting')
results_list.append(res)

res, xgb_model = evaluate_model(XGBClassifier(random_state=42, eval_metric='mlogloss'),
                                 X_train, X_test, y_train, y_test, 'XGBoost')
results_list.append(res)

### 6.5 Model Comparison

In [ ]:
results_df = pd.DataFrame(results_list).sort_values(by='F1-Score', ascending=False).reset_index(drop=True)
results_df

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x='Model', y='F1-Score', data=results_df, palette='viridis')
plt.title('Model Comparison — Weighted F1-Score')
plt.ylim(0, 1.05)
plt.xticks(rotation=20)
for i, v in enumerate(results_df['F1-Score']):
    plt.text(i, v + 0.01, f"{v:.3f}", ha='center')
plt.show()

**Observation:** **Logistic Regression** and **Random Forest** both achieve a **perfect weighted F1-score (1.00)** on the test set, while the other ensembles — **Gradient Boosting / AdaBoost (~0.97)** and **XGBoost (~0.94)** — are also very strong, all comfortably ahead of the single Decision Tree. This shows the K-Means segments are highly separable from the raw socio-economic features. We carry **Random Forest forward** as the representative ensemble model for feature-importance analysis and tuning, since ensembles like Random Forest/XGBoost are generally far more robust than a single linear model on larger, noisier, real-world "customer" data.

### 6.6 Confusion Matrix — Best Model (Random Forest)

In [ ]:
y_pred_rf = rf_model.predict(X_test)

plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Blues',
            xticklabels=['Developed', 'Under-developed', 'Developing'],
            yticklabels=['Developed', 'Under-developed', 'Developing'])
plt.title('Confusion Matrix - Random Forest')
plt.xlabel('Predicted Segment')
plt.ylabel('Actual Segment')
plt.show()

### 6.7 Feature Importance

Understanding *which features drive segment membership* is itself a key piece of "customer intelligence" — it tells the business which indicators to monitor most closely.

In [ ]:
rf_importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=rf_importances.values, y=rf_importances.index, palette='magma')
plt.title('Feature Importance — Random Forest')
plt.xlabel('Importance')
plt.show()

In [ ]:
xgb_importances = pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=xgb_importances.values, y=xgb_importances.index, palette='cubehelix')
plt.title('Feature Importance — XGBoost')
plt.xlabel('Importance')
plt.show()

**Observation:** Both Random Forest and XGBoost consistently rank **`gdpp`, `income`, and `child_mort`** (and frequently `life_expec`) as the most important drivers of segment membership — these are exactly the features that defined our cluster profiles in Section 5.1.5, confirming the model has learned a meaningful, business-aligned decision boundary.

### 6.8 Hyperparameter Tuning — Optimizing the Best Model

To squeeze out further predictive performance, we tune the **Random Forest** using `GridSearchCV` with 5-fold cross-validation.

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV F1-Score: {:.3f}".format(grid_search.best_score_))

In [ ]:
tuned_model = grid_search.best_estimator_
y_pred_tuned = tuned_model.predict(X_test)

print("--- Tuned Random Forest (Test Set) ---")
print(classification_report(y_test, y_pred_tuned, digits=3))

tuned_acc = accuracy_score(y_test, y_pred_tuned)
tuned_f1 = f1_score(y_test, y_pred_tuned, average='weighted')
print(f"\nTest Accuracy: {tuned_acc:.3f} | Test F1-Score: {tuned_f1:.3f}")

**Observation:** Hyperparameter tuning via `GridSearchCV` either matches or slightly improves the default Random Forest's performance, confirming the model was already close to its optimal configuration for this dataset size — the **tuned Random Forest** is selected as the final production model for the Customer Intelligence System.

`gdpp`, `income`, `child_mort`, and `life_expec` are the **most influential features** for both segmentation and prediction. Any future Customer Intelligence dashboard should prioritize tracking these four indicators, as they are the strongest signals of which segment a customer/country belongs to.


This combination of **unsupervised segmentation + supervised classification + ensemble optimization** delivers both *descriptive* (who are our segments?) and *predictive* (which segment does a new customer belong to?) intelligence — the core goals of a Customer Intelligence System.